In [ ]:
from pathlib import Path
import sys
parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))
from config import BIGQUERY_API_KEY
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError

import hashlib
import json
import time
from datetime import datetime, timedelta
import pandas as pd
import requests
from bs4 import BeautifulSoup

client = bigquery.Client(project="proven-reality-499800-u9")



In [ ]:
# initialize everything

from pathlib import Path

SPONSORS = ["Progressive", "Cheddar's Scratch Kitchen", "Love's Travel Stops", "Busch Light", "Castrol"]

SEASONS = [
    ("2025-02-01", "2025-12-01"),
    ("2026-02-01", "2026-12-01"),
]

DOC_API_ROLLING_DAYS = 85

SPORTS_DOMAINS = [
    "nascar.com", "jayski.com", "frontstretch.com", "foxsports.com",
    "espn.com", "si.com", "motorsport.com", "autoweek.com", "racer.com",
    "nbcsports.com", "usatoday.com", "cbssports.com", "bleacherreport.com",
    "athlonsports.com", "sportsnaut.com", "motorsportweek.com",
    "speedwaydigest.com",
    # added to catch NASCAR-specific trade outlets missed by general sports sites
    # (confirmed via research to carry sponsor-announcement stories, e.g. for Cheddar's Scratch Kitchen)
    "tobychristie.com", "speedwaymedia.com", "racingamerica.com",
]

# bump this whenever SPORTS_DOMAINS (or other candidate-query logic) changes,
# so previously-cached months are re-fetched instead of silently reused
CACHE_VERSION = "v2"

URL_KEYWORDS = ["nascar", "cup-series", "xfinity-series", "truck-series", "daytona", "talladega"]

OUTPUT_CSV = Path("data/raw/sponsor_mentions.csv")
CHECKPOINT_FILE = Path("sponsor_mentions_checkpoint.json")
FAILED_LOG = Path("sponsor_mentions_failed.json")
CANDIDATE_CACHE_DIR = Path("candidate_cache")   # one JSON per month of BQ candidate URLs
ARTICLE_TEXT_CACHE_DIR = Path("article_text_cache")  # one .txt per scraped URL

DOC_API_URL = "https://api.gdeltproject.org/api/v2/doc/doc"
REQUEST_HEADERS = {"User-Agent": "Mozilla/5.0 (research script; contact: lltodaro@gmail.com)"}

CANDIDATE_CACHE_DIR.mkdir(exist_ok=True)
ARTICLE_TEXT_CACHE_DIR.mkdir(exist_ok=True)

MAX_GB_WARNING = 5.0  # print a warning if a single query scans more than this

DOC_API_DELAY_SECONDS = 6


QUERY = """
    SELECT
      DATE,
      DocumentIdentifier AS url,
      SourceCommonName AS source,
      V2Organizations,
      V2Tone
    FROM `gdelt-bq.gdeltv2.gkg_partitioned`
    WHERE _PARTITIONTIME >= TIMESTAMP(@start_date)
      AND _PARTITIONTIME <  TIMESTAMP(@end_date)
      AND V2Organizations LIKE @sponsor_pattern
      AND (LOWER(V2Organizations) LIKE '%nascar%' OR LOWER(AllNames) LIKE '%nascar%')
"""


In [ ]:
def month_windows(start_str, end_str):
    start = pd.Timestamp(start_str)
    end = pd.Timestamp(end_str)
    windows = []
    cur = start
    while cur < end:
        nxt = min(cur + pd.offsets.MonthBegin(1), end)
        windows.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt
    return windows

In [ ]:
import json
def load_checkpoint():
    """Load the set of (sponsor, start, end) jobs that already completed successfully."""
    if CHECKPOINT_FILE.exists():
        return set(tuple(x) for x in json.loads(CHECKPOINT_FILE.read_text()))
    return set()

 
def save_checkpoint(done):
    CHECKPOINT_FILE.write_text(json.dumps([list(x) for x in done]))
    
def log_failure(sponsor, start, end, error):
    failures = []
    if FAILED_LOG.exists():
        failures = json.loads(FAILED_LOG.read_text())
    failures.append({"sponsor": sponsor, "start": start, "end": end, "error": str(error)})
    FAILED_LOG.write_text(json.dumps(failures, indent=2))
 
 
def _csv_has_content():
    return OUTPUT_CSV.exists() and OUTPUT_CSV.stat().st_size > 0


def append_results(rows):
    """Append new mention rows to OUTPUT_CSV, skipping any (sponsor, url) pair that's already
    in the file - so reprocessing a window (e.g. after a checkpoint reset) can never create
    duplicate rows."""
    if not rows:
        return
    df = pd.DataFrame(rows)

    if _csv_has_content():
        existing = pd.read_csv(OUTPUT_CSV, usecols=["sponsor", "url"])[["sponsor", "url"]]
        existing_keys = set(map(tuple, existing.itertuples(index=False)))
        df = df[~df.apply(lambda r: (r["sponsor"], r["url"]) in existing_keys, axis=1)]
        if df.empty:
            return

    write_header = not _csv_has_content()
    df.to_csv(OUTPUT_CSV, mode="a", header=write_header, index=False)
    
def url_hash(url):
    return hashlib.sha256(url.encode("utf-8")).hexdigest()[:24]

In [ ]:
import re

def normalize(text):
    """Lowercase and strip apostrophe variants so 'Cheddar's' matches "Cheddar's"."""
    text = text.lower()
    text = re.sub(r"[‘’ʼ'`]", "", text)  # curly/straight apostrophes -> nothing
    return text

# Map each sponsor to one or more short search terms that are likely to actually appear in text
SPONSOR_KEYWORDS = {
    "Cheddar's Scratch Kitchen": ["cheddars scratch kitchen", "cheddars 300", "cheddars"],
    "Love's Travel Stops": ["loves travel stops", "loves 500", "loves rv stop"],
    "Progressive": ["Progressive"],
    "Busch Light": ["busch light"],
    "Castrol": ["castrol"],
}

# Sponsors whose keyword is also an ordinary English word/adjective ("progressive banking",
# "progressively") need a stricter case-sensitive whole-word match - a normalized substring
# match on these picks up heavy false-positive noise from the generic word. Everything else
# is distinctive enough that normalize()+substring is fine.
CASE_SENSITIVE_WHOLE_WORD_SPONSORS = {"Progressive"}

def sponsor_mentioned(sponsor, text):
    """True if `text` contains a real mention of `sponsor`, per SPONSOR_KEYWORDS."""
    if sponsor in CASE_SENSITIVE_WHOLE_WORD_SPONSORS:
        return any(re.search(rf"\b{re.escape(kw)}\b", text) for kw in SPONSOR_KEYWORDS[sponsor])
    norm_text = normalize(text)
    return any(normalize(kw) in norm_text for kw in SPONSOR_KEYWORDS[sponsor])

In [ ]:
# recent dates:

def query_doc_api(sponsor, start, end, max_retries=4):
    """Full-text search via GDELT DOC 2.0 API. Returns a list of article dicts."""
    start_dt = start.replace("-", "") + "000000"
    end_dt = end.replace("-", "") + "235959"
 
    params = {
        "query": f'"{sponsor}" NASCAR',
        "mode": "artlist",
        "format": "json",
        "maxrecords": 250,
        "sort": "datedesc",
        "STARTDATETIME": start_dt,
        "ENDDATETIME": end_dt,
    }
    
    for attempt in range(max_retries):
        resp = requests.get(DOC_API_URL, params=params, headers=REQUEST_HEADERS, timeout=30)
        if resp.status_code == 429:
            wait = 10 * (2 ** attempt)  # 10s, 20s, 40s, 80s
            print(f"    rate limited, waiting {wait}s before retry {attempt + 1}/{max_retries}")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        break
    else:
        raise RuntimeError(f"Failed to query DOC API after {max_retries} attempts")
    data = resp.json()
    articles = data.get("articles", [])
 
    if len(articles) == 250:
        print(f"    note: hit the 250-record cap for {sponsor} {start} to {end} - "
              f"consider splitting this window smaller if completeness matters")
 
    rows = []
    for a in articles:
        rows.append({
            "sponsor": sponsor,
            "url": a.get("url"),
            "title": a.get("title"),
            "domain": a.get("domain"),
            "seendate": a.get("seendate"),
            "source_method": "doc_api",
        })
    return rows
 

In [ ]:
# old dates via bigquery:

def get_candidate_urls(client, start, end):
    """Get (and cache) candidate article URLs from known sports domains for one month."""
    cache_file = CANDIDATE_CACHE_DIR / f"{start}_{end}_{CACHE_VERSION}.json"
    if cache_file.exists():
        return json.loads(cache_file.read_text())
 
    query = """
        SELECT DISTINCT DocumentIdentifier AS url, SourceCommonName AS domain, DATE
        FROM `gdelt-bq.gdeltv2.gkg_partitioned`
        WHERE _PARTITIONTIME >= TIMESTAMP(@start_date)
          AND _PARTITIONTIME <  TIMESTAMP(@end_date)
          AND SourceCommonName IN UNNEST(@domains)
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("start_date", "STRING", start),
            bigquery.ScalarQueryParameter("end_date", "STRING", end),
            bigquery.ArrayQueryParameter("domains", "STRING", SPORTS_DOMAINS),
        ]
    )
 
    dry_config = bigquery.QueryJobConfig(
        dry_run=True, use_query_cache=False, query_parameters=job_config.query_parameters,
    )
    dry_job = client.query(query, job_config=dry_config)
    gb_scanned = dry_job.total_bytes_processed / 1e9
    if gb_scanned > MAX_GB_WARNING:
        print(f"    warning: candidate query for {start} to {end} will scan {gb_scanned:.2f} GB")
 
    df = client.query(query, job_config=job_config).to_dataframe()
    candidates = df.to_dict("records")
    candidates = [c for c in candidates if is_likely_nascar_url(c["url"])]
    cache_file.write_text(json.dumps(candidates, default=str))
    return candidates


def fetch_article_text(url):
    """Fetch and cache the visible text of a URL. Returns '' on failure.

    IMPORTANT: successful-but-empty fetches are cached as `<hash>.txt` (empty file).
    Failed fetches (network errors, timeouts, 4xx/5xx) are cached separately as
    `<hash>.failed.json` so they are NOT confused with a real 'page fetched fine,
    sponsor just isn't mentioned' result, and so they can be identified/retried later
    (see summarize_fetch_failures() / retry_failed_fetches() below).
    """
    cache_file = ARTICLE_TEXT_CACHE_DIR / f"{url_hash(url)}.txt"
    fail_file = ARTICLE_TEXT_CACHE_DIR / f"{url_hash(url)}.failed.json"

    if cache_file.exists():
        return cache_file.read_text(encoding="utf-8", errors="ignore")
    if fail_file.exists():
        return ""  # previously failed; skip re-fetching until retry_failed_fetches() is run

    try:
        resp = requests.get(url, headers=REQUEST_HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "header", "footer"]):
            tag.decompose()
        text = soup.get_text(separator=" ", strip=True)
        cache_file.write_text(text, encoding="utf-8")
        return text
    except Exception as e:
        status_code = getattr(getattr(e, "response", None), "status_code", None)
        fail_file.write_text(json.dumps({
            "url": url,
            "error": str(e),
            "status_code": status_code,
        }))
        return ""


def scan_candidates_for_sponsor(sponsor, candidates):
    """Search cached/fetched article text for a sponsor mention."""
    rows = []
    for c in candidates:
        
        text = fetch_article_text(c["url"])
        if text and sponsor_mentioned(sponsor, text):
            rows.append({
                "sponsor": sponsor,
                "url": c["url"],
                "title": None,
                "domain": c.get("domain"),
                "seendate": c.get("DATE"),
                "source_method": "bq_candidate_scrape",
            })
        time.sleep(0.2)  # be polite between page fetches
    return rows

def is_likely_nascar_url(url):
    url_lower = url.lower()
    return any(keyword in url_lower for keyword in URL_KEYWORDS)

In [ ]:
# Diagnostics for the "zero mentions for everything" symptom.
# Run summarize_fetch_failures() BEFORE a full rerun to see whether matches are coming
# back empty because pages genuinely don't mention the sponsor, or because requests to
# scrape them are failing (bot-blocking, timeouts, etc.) and silently returning no text.

from collections import Counter

def summarize_fetch_failures():
    ok_files = list(ARTICLE_TEXT_CACHE_DIR.glob("*.txt"))
    failed_files = list(ARTICLE_TEXT_CACHE_DIR.glob("*.failed.json"))
    empty_ok = [f for f in ok_files if not f.read_text(encoding="utf-8", errors="ignore").strip()]

    print(f"successfully fetched: {len(ok_files)} ({len(empty_ok)} came back with no visible text at all)")
    print(f"failed to fetch:      {len(failed_files)}")

    if not failed_files:
        return

    reasons = Counter()
    status_codes = Counter()
    for f in failed_files:
        try:
            payload = json.loads(f.read_text())
        except Exception:
            reasons["unreadable failure record"] += 1
            continue
        reasons[payload.get("error", "unknown").split(":")[0]] += 1
        status_codes[payload.get("status_code")] += 1

    print("\ntop failure reasons:")
    for reason, count in reasons.most_common(10):
        print(f"  {count:5d}  {reason}")

    print("\nstatus codes seen:")
    for code, count in status_codes.most_common(10):
        print(f"  {count:5d}  {code}")


def retry_failed_fetches():
    """Delete failure markers so those URLs get re-attempted on the next scan.
    Run this after changing REQUEST_HEADERS or otherwise addressing why fetches failed -
    otherwise the same URLs will just fail again for the same reason."""
    failed_files = list(ARTICLE_TEXT_CACHE_DIR.glob("*.failed.json"))
    for f in failed_files:
        f.unlink()
    print(f"cleared {len(failed_files)} failure markers - they'll be retried next run")


def spot_check_url(url):
    """Fetch one URL fresh (bypassing both caches) and print status/snippet, to sanity-check
    a single known article by hand."""
    resp = requests.get(url, headers=REQUEST_HEADERS, timeout=15)
    print("status:", resp.status_code)
    print("retry-after header:", resp.headers.get("Retry-After"))
    print("body snippet:", resp.text[:500])
    return resp


In [ ]:
# Force re-scraping of windows the checkpoint previously marked "done".
#
# Bumping CACHE_VERSION made get_candidate_urls() re-query BigQuery for a fresh candidate
# list (now including the new trade-outlet domains) - but the checkpoint file is keyed by
# (sponsor, start, end, method), which is unrelated to the candidate cache version. Any
# window already marked "done" from a prior run will still be skipped by the main loop
# and will never see the new candidates. Clear the relevant entries before rerunning.
#
# This only clears the checkpoint (i.e. which windows get reprocessed) - it does NOT
# delete cached article text, so URLs you've already scraped successfully won't be
# re-fetched over the network; only genuinely new candidate URLs will be.

def clear_checkpoint(method=None, sponsor=None):
    """Remove checkpoint entries matching the given filters (None = match anything),
    forcing those (sponsor, start, end, method) jobs to be reprocessed on the next run.

    Handles a mix of tuple shapes: some checkpoint entries are legacy 3-tuples
    (sponsor, start, end) written before the `method` field existed - those all came
    from the original bq-scrape-only pipeline, so they're treated as method='bq_scrape'
    for filtering purposes.
    """
    done = load_checkpoint()
    before = len(done)
    kept = set()
    skipped_unrecognized = 0
    for entry in done:
        if len(entry) == 4:
            entry_sponsor, entry_start, entry_end, entry_method = entry
        elif len(entry) == 3:
            entry_sponsor, entry_start, entry_end = entry
            entry_method = "bq_scrape"  # legacy entry, predates the method field
        else:
            kept.add(entry)  # unrecognized shape - leave it alone rather than guess
            skipped_unrecognized += 1
            continue

        if method is not None and entry_method != method:
            kept.add(entry)
            continue
        if sponsor is not None and entry_sponsor != sponsor:
            kept.add(entry)
            continue
        # matched all given filters -> drop it (i.e. force reprocessing)
    save_checkpoint(kept)
    print(f"cleared {before - len(kept)} checkpoint entries; {len(kept)} remain")
    if skipped_unrecognized:
        print(f"note: left {skipped_unrecognized} entries with an unexpected shape untouched")

# Clear every historical (bq_scrape) window for every sponsor, since the new domains
# affect all of them. doc_api entries (recent months) are untouched since that method
# doesn't depend on SPORTS_DOMAINS at all.
clear_checkpoint(method="bq_scrape")


In [14]:
# The main loop above no longer has a doc_api/is_recent split - every window is processed the
# same way (real BigQuery candidates + real scan) once it has fully elapsed. That split used to
# tag near-term windows as method="doc_api" and mark them "done" with zero results as soon as
# query_doc_api() was disabled. Clear those stale entries so the real (sponsor, window) jobs
# they were standing in for get processed for real on the next run.
clear_checkpoint(method="doc_api")


cleared 40 checkpoint entries; 60 remain


In [15]:
client = bigquery.Client(project="proven-reality-499800-u9")
done = load_checkpoint()

all_windows = []
for season_start, season_end in SEASONS:
    all_windows.extend(month_windows(season_start, season_end))

# Only fully-elapsed months have anything to find. Windows still in progress or entirely in
# the future are left alone (NOT marked done) so they get picked up automatically on a later
# run once they've actually happened. This replaces the old DOC_API_ROLLING_DAYS cutoff, which
# tagged near-term windows as method="doc_api" and silently "completed" them with an empty
# candidate list once the query_doc_api() call itself was disabled below - that's what was
# quietly producing zero rows for the last several months of the season.
now = datetime.utcnow()
ended_windows = [w for w in all_windows if pd.Timestamp(w[1]) <= now]
future_windows = [w for w in all_windows if w not in ended_windows]
if future_windows:
    print(f"skipping {len(future_windows)} window(s) that haven't fully elapsed yet "
          f"(will be picked up on a future run once they have):")
    for w in future_windows:
        print("   ", w)

# Pre-fetch candidate URLs once per elapsed month (shared across all sponsors)
candidates_by_window = {}
for start, end in ended_windows:
    print(f"fetching BigQuery candidates for {start} to {end}")
    try:
        candidates_by_window[(start, end)] = get_candidate_urls(client, start, end)
    except GoogleAPIError as e:
        print(f"    failed: {e}")
        log_failure("ALL", start, end, e)
        candidates_by_window[(start, end)] = []

total_jobs = len(SPONSORS) * len(ended_windows)
job_num = 0

for sponsor in SPONSORS:
    for start, end in ended_windows:
        job_num += 1
        method = "bq_scrape"
        key = (sponsor, start, end, method)

        if key in done:
            print(f"[{job_num}/{total_jobs}] skip  {sponsor} {start} to {end} (already done)")
            continue

        print(f"[{job_num}/{total_jobs}] {sponsor} {start} to {end}")
        try:
            candidates = candidates_by_window.get((start, end), [])
            rows = scan_candidates_for_sponsor(sponsor, candidates)
        except Exception as e:
            print(f"    failed: {e}")
            log_failure(sponsor, start, end, e)
            continue

        append_results(rows)
        print(f"    found {len(rows)} mentions")

        done.add(key)
        save_checkpoint(done)
        time.sleep(0.3)

print("\nDone. Results in:", OUTPUT_CSV.resolve())
if FAILED_LOG.exists():
    print("Some queries failed - see:", FAILED_LOG.resolve())


/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


skipping 5 window(s) that haven't fully elapsed yet (will be picked up on a future run once they have):
    ('2026-07-01', '2026-08-01')
    ('2026-08-01', '2026-09-01')
    ('2026-09-01', '2026-10-01')
    ('2026-10-01', '2026-11-01')
    ('2026-11-01', '2026-12-01')
fetching BigQuery candidates for 2025-02-01 to 2025-03-01
fetching BigQuery candidates for 2025-03-01 to 2025-04-01
fetching BigQuery candidates for 2025-04-01 to 2025-05-01
fetching BigQuery candidates for 2025-05-01 to 2025-06-01
fetching BigQuery candidates for 2025-06-01 to 2025-07-01
fetching BigQuery candidates for 2025-07-01 to 2025-08-01
fetching BigQuery candidates for 2025-08-01 to 2025-09-01
fetching BigQuery candidates for 2025-09-01 to 2025-10-01
fetching BigQuery candidates for 2025-10-01 to 2025-11-01
fetching BigQuery candidates for 2025-11-01 to 2025-12-01
fetching BigQuery candidates for 2026-02-01 to 2026-03-01
fetching BigQuery candidates for 2026-03-01 to 2026-04-01
fetching BigQuery candidates for 20

/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


fetching BigQuery candidates for 2026-05-01 to 2026-06-01


/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


fetching BigQuery candidates for 2026-06-01 to 2026-07-01


/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


[1/75] skip  Progressive 2025-02-01 to 2025-03-01 (already done)
[2/75] skip  Progressive 2025-03-01 to 2025-04-01 (already done)
[3/75] skip  Progressive 2025-04-01 to 2025-05-01 (already done)
[4/75] skip  Progressive 2025-05-01 to 2025-06-01 (already done)
[5/75] skip  Progressive 2025-06-01 to 2025-07-01 (already done)
[6/75] skip  Progressive 2025-07-01 to 2025-08-01 (already done)
[7/75] skip  Progressive 2025-08-01 to 2025-09-01 (already done)
[8/75] skip  Progressive 2025-09-01 to 2025-10-01 (already done)
[9/75] skip  Progressive 2025-10-01 to 2025-11-01 (already done)
[10/75] skip  Progressive 2025-11-01 to 2025-12-01 (already done)
[11/75] skip  Progressive 2026-02-01 to 2026-03-01 (already done)
[12/75] skip  Progressive 2026-03-01 to 2026-04-01 (already done)
[13/75] Progressive 2026-04-01 to 2026-05-01
    found 2 mentions
[14/75] Progressive 2026-05-01 to 2026-06-01
    found 1 mentions
[15/75] Progressive 2026-06-01 to 2026-07-01
    found 1 mentions
[16/75] skip  Chedd

In [ ]:
import sys
print(sys.executable)

In [ ]:
%pip install db-dtypes

In [ ]:
sql = """
SELECT COUNT(*) as cnt
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("2025-06-01")
  AND _PARTITIONTIME <  TIMESTAMP("2025-07-01")
  AND LOWER(V2Organizations) LIKE '%nascar%'
"""

df = client.query(sql).to_dataframe()
print(df)

In [ ]:
sql = """
SELECT COUNT(*) as cnt
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("2025-06-01")
  AND _PARTITIONTIME <  TIMESTAMP("2025-07-01")
  AND LOWER(V2Organizations) LIKE '%google%'
"""
df = client.query(sql).to_dataframe()
print(df)

In [ ]:
import json
from pathlib import Path
import pandas as pd

CHECKPOINT_FILE = Path("sponsor_mentions_checkpoint.json")
CANDIDATE_CACHE_DIR = Path("candidate_cache")

# Recreate the same window list your main script uses
def month_windows(start_str, end_str):
    start = pd.Timestamp(start_str)
    end = pd.Timestamp(end_str)
    windows = []
    cur = start
    while cur < end:
        nxt = min(cur + pd.offsets.MonthBegin(1), end)
        windows.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt
    return windows

SEASONS = [
    ("2025-02-01", "2025-12-01"),
    ("2026-02-01", "2026-12-01"),
]

all_windows = []
for season_start, season_end in SEASONS:
    all_windows.extend(month_windows(season_start, season_end))

# Find which windows genuinely have a candidate cache file (real, already-queried months)
windows_with_cache = set()
for start, end in all_windows:
    cache_file = CANDIDATE_CACHE_DIR / f"{start}_{end}.json"
    if cache_file.exists():
        windows_with_cache.add((start, end))

windows_missing_cache = [w for w in all_windows if w not in windows_with_cache]
print(f"{len(windows_missing_cache)} windows never actually got candidates fetched:")
for w in windows_missing_cache:
    print("  ", w)

# Load checkpoint, drop any entry whose (start, end) falls in the "never fetched" set
done = set(tuple(x) for x in json.loads(CHECKPOINT_FILE.read_text()))
before_count = len(done)

cleaned = {
    entry for entry in done
    if not (entry[1], entry[2]) in windows_missing_cache
}

print(f"Removed {before_count - len(cleaned)} bogus checkpoint entries")
CHECKPOINT_FILE.write_text(json.dumps([list(x) for x in cleaned]))

In [ ]:
from pathlib import Path

cache_files = list(Path("article_text_cache").glob("*.txt"))
lengths = [len(f.read_text(encoding="utf-8", errors="ignore")) for f in cache_files]

print(f"{len(cache_files)} cached articles")
print(f"average length: {sum(lengths)/len(lengths):.0f} characters")
print(f"shortest 5: {sorted(lengths)[:5]}")

In [ ]:
from pathlib import Path

sample_files = list(Path("article_text_cache").glob("*.txt"))[:20]
sponsors_to_test = SPONSORS

for f in sample_files:
    text = f.read_text(encoding="utf-8", errors="ignore").lower()
    if len(text) < 100:
        continue
    hits = [s for s in sponsors_to_test if s.lower() in text]
    print(f.name, "->", hits if hits else "no matches")

In [ ]:
from pathlib import Path

all_files = list(Path("article_text_cache").glob("*.txt"))

for f in all_files:
    text = f.read_text(encoding="utf-8", errors="ignore").lower()
    if "cheddar" in text:
        idx = text.find("cheddar")
        print(f.name)
        print(text[max(0, idx-150):idx+150])
        print("---")

In [ ]:
# empty the csv to reset the results
filename = Path("data/raw/sponsor_mentions.csv")
with open(filename, "w") as file:
    pass  

In [17]:
# make a dataframe
news_df = pd.read_csv("data/raw/sponsor_mentions.csv")

In [20]:
# Check for duplicate entries
duplicates = news_df.duplicated(subset=['sponsor', 'url'])
if duplicates.sum() > 0:
    print(f"Warning: {duplicates.sum()} duplicate articles found")
    print(news_df[duplicates])

In [ ]:
# Verify source variety
print("\nMention sources:")
print(news_df['domain'].value_counts().head(15))

print("\nMention types:")
print(news_df['Mention_Type'].value_counts())